# Financial Copilot: fine-tune Llama 3.1 8B with QLoRA

This notebook trains the "receptionist" half of Financial Copilot: a Llama 3.1 8B model that reads a money question, picks one of 38 calculators and fills in the numbers. Python does the math.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is free; L4 or A100 is much faster). Then run the cells top to bottom.

What you'll end up with:
1. A LoRA adapter (about 170 MB) saved to your Google Drive
2. An evaluation report comparing the fine-tuned model with the untuned base model on questions it never saw
3. Optionally, a GGUF file to run the model on your laptop with Ollama

Rough timing: setup about 5 minutes; training depends on the GPU Colab assigns (a free T4 takes a few hours for the full dataset, an A100 well under an hour); the progress bar shows a live estimate.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install Unsloth
Unsloth makes QLoRA training about twice as fast and uses much less memory. This takes 2–3 minutes.

In [ ]:
%%capture
!pip install unsloth

## 3. Get the project code
Either upload the `financial-copilot.zip` you downloaded, or set `REPO_URL` to your GitHub copy.

In [ ]:
import os, zipfile

REPO_URL = ""  # e.g. "https://github.com/<your-username>/financial-copilot"; leave empty to upload the zip

if not os.path.exists("/content/financial-copilot"):
    if REPO_URL:
        !git clone {REPO_URL} /content/financial-copilot
    else:
        from google.colab import files
        uploaded = files.upload()  # choose financial-copilot.zip
        zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
%cd /content/financial-copilot

## 4. Save to Google Drive (recommended)
Free Colab sessions can disconnect. Saving checkpoints to Drive means you can resume instead of starting over.

In [ ]:
USE_DRIVE = True
OUT = "outputs/fincopilot-lora"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/financial-copilot/fincopilot-lora"
print("The adapter will be saved to", OUT)

## 5. Test the code and build the dataset
The tests check every calculator, the agent loop and the data generator (about 100 tests, a few seconds). Then we generate the ~7.8k training conversations and audit every label.

In [ ]:
!python -m unittest discover -s tests -t . 2>&1 | tail -3
!python -m fincopilot.datagen.generate --out data
!python -m fincopilot.datagen.validate data

## 6. Look at the tokenized data
This loads only the tokenizer. It prints token lengths and one example where the text inside `[[ ]]` is what the model learns. Notice that the system prompt, your question and the tool's result are *not* inside the brackets: the model is never trained to predict numbers that Python produces.

In [ ]:
!python training/train.py --dry-run

## 7. Train
Defaults: LoRA rank 16, alpha 32, learning rate 2e-4, 1 epoch, effective batch size 16 (about 490 steps).

* Out of memory on a T4? Add `--batch-size 1 --grad-accum 16`.
* Session disconnected? Re-run cells 2–4, then run this cell again with `--resume` added.
* Want a 5-minute smoke test first? Use the commented line instead.

In [ ]:
!python training/train.py --out "{OUT}"
# !python training/train.py --out "{OUT}" --limit 800 --max-steps 60

## 8. Evaluate: fine-tuned vs base model
Each question gets a single reply with no self-correction, so this measures the model itself. The base model gets the long prompt with every tool described (its best chance); the fine-tuned model gets the short one.

`--limit 400` samples the 1,092-question held-out set (every question type is kept) and always runs all 94 hand-written real-world questions. Remove `--limit` for the full set.

In [ ]:
!python -m fincopilot.evaluation --backend hf --adapter "{OUT}" --compare-base --limit 400 --explain --out reports

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("reports/comparison.md").read()))

## 9. Try it

In [ ]:
from fincopilot.agent import Agent
from fincopilot.backends.hf import HFBackend

backend = HFBackend(adapter=OUT)
agent = Agent(backend)
history = []

def ask(question):
    global history
    reply = agent.chat(history, question)
    history = reply.history
    for i, step in enumerate(reply.steps, 1):
        print(f"{i}. {step.stage:<10} {step.status:<6} {step.detail}")
    print("\n" + reply.text)

ask("If I invest $10,000 at 7% for 10 years, how much will I have?")

In [ ]:
ask("What would my monthly payment be on a $400,000 house?")   # it should ask for the missing details

In [ ]:
ask("6.5% for 30 years, with 20% down")   # answers using the earlier context

### Optional: the web app inside Colab
Opens the same interface you get locally, served from this notebook.

In [ ]:
import threading
from google.colab import output
from fincopilot.web.server import create_app

app = create_app(agent, "Fine-tuned Llama 3.1 8B via Hugging Face (Colab)")
threading.Thread(target=lambda: app.run(port=7860, threaded=False), daemon=True).start()
output.serve_kernel_port_as_window(7860)

## 10. Export for your laptop (Ollama)
Merges the adapter into the base model and converts it to a 4-bit GGUF (about 5 GB) plus an Ollama `Modelfile`. This needs GPU memory, so free the demo model first. Exporting takes 10–20 minutes.

In [ ]:
import gc, torch
for name in ("agent", "backend", "app"):
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()
!python training/export_gguf.py --adapter "{OUT}" --out /content/fincopilot --quant q4_k_m

In [ ]:
# Copy the GGUF and Modelfile to Drive (faster than downloading 5 GB through the browser).
import glob, shutil, os
dest = "/content/drive/MyDrive/financial-copilot/ollama"
os.makedirs(dest, exist_ok=True)
for path in glob.glob("/content/fincopilot*/**/*.gguf", recursive=True) + glob.glob("/content/fincopilot*/**/Modelfile", recursive=True):
    shutil.copy(path, dest)
    print("copied", os.path.basename(path))
print("Download the folder", dest, "from Google Drive, then run: ollama create fincopilot -f Modelfile")

## 11. Optional: publish the adapter on Hugging Face
Makes your model public so others can try it. You need a free account and a write token from https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import HfApi, login
login()
REPO_ID = "<your-username>/fincopilot-llama-3.1-8b-lora"
api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=REPO_ID, ignore_patterns=["checkpoint-*"])
print("Published: https://huggingface.co/" + REPO_ID)

## Your results
Open `reports/comparison.md` (saved in the `reports` folder). The **answer accuracy** row is the share of calculation questions, phrased in ways the model never saw in training, where its tool call produced exactly the right result. The hand-written real-world set is the best guide to how it handles real users.